In [1]:
import pandas as pd
import numpy as np
import mysql.connector

In [2]:
DB_NAME = 'accidents'

# 1) อ่าน CSV — index_col=0 เพื่อทิ้งคอลัมน์ index ที่เกินมา
df = pd.read_csv("airplane_crashes_cleaned.csv", index_col=0)

# 2) แปลง NaN -> None  (MySQL จะเก็บเป็น NULL)
df = df.replace({np.nan: None})

In [ ]:
# 3) เชื่อมต่อ MySQL
cnx = mysql.connector.connect(user='root', password='', host='localhost', database=DB_NAME)
cursor = cnx.cursor()

# 4) (ตัวเลือก) ล้างตารางก่อน เพื่อให้รันซ้ำได้โดยไม่เกิดข้อมูลซ้ำ
cursor.execute("TRUNCATE TABLE accidents")

# 5) เตรียมคำสั่ง INSERT — ลำดับคอลัมน์ตรงกับ CSV เป๊ะ (20 คอลัมน์)
insert_sql = (
    "INSERT INTO accidents "
    "(date, time, location, operator, flight_no, route, ac_type, registration, cn_ln, "
    " aboard, fatalities, ground, summary, year, month, decade, survivors, "
    " survival_rate, fatality_rate, country) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, "
    "        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"
)

# 6) แปลง DataFrame เป็น list of tuples แล้ว insert ทีเดียว (เร็วกว่า loop)
rows = list(df.itertuples(index=False, name=None))
# Insert in smaller batches to avoid exceeding MySQL's max_allowed_packet
batch_size = 100
total_inserted = 0

for start in range(0, len(rows), batch_size):
    batch = rows[start:start + batch_size]
    cursor.executemany(insert_sql, batch)
    cnx.commit()
    total_inserted += cursor.rowcount

print(f"Data loaded successfully: {total_inserted} rows inserted.")

cursor.close()
cnx.close()

โหลดสำเร็จ: 4963 แถว
